# Symmetric encryption and key management

# Coding problem, 5.3.b: Symmetric key-management toolkit

**Skill:** Mitigate Risk · **Maps to:** 5.3.A, 5.3.B · **Language:** Python 3 (pytest)

## Problem
Build three small helpers a security engineer would use when reasoning about symmetric key
management, without relying on any real crypto library (standard library only, this is a toy,
pedagogical model, not production cryptography).

1. **Key wrapping (envelope encryption).** Implement `wrap_key(dek, kek)` and
   `unwrap_key(wrapped, kek)` using a repeating-XOR keystream: each byte of `dek` is XORed with
   the KEK byte at the same position modulo `len(kek)`. This models how a Key Encryption Key
   (KEK) wraps a Data Encryption Key (DEK) for storage (see atom.md, "Key wrapping (Envelope
   Encryption)"). Unwrapping with the same KEK must recover the original DEK exactly, since
   XOR is its own inverse.
2. **Weak-RNG detection.** Implement `looks_like_csprng(key_bytes)`, a heuristic that flags keys
   that could NOT plausibly have come from a CSPRNG because they show low entropy / obvious
   structure (e.g., a single repeated byte, a short repeating pattern, or a strictly
   monotonic/arithmetic byte sequence like `0,1,2,3...`). Real CSPRNG output has no such
   structure.
3. **ECB pattern-leakage detection.** Implement `detect_ecb_leakage(ciphertext_blocks)`, which
   takes a list of same-size `bytes` blocks (as an ECB-mode block cipher would produce) and
   returns `True` if any two blocks are identical, the tell-tale sign that ECB mode is leaking
   repeated plaintext patterns into the ciphertext (see atom.md, "Electronic Codebook (ECB) Mode
   (Insecure)").

## Function contract
Implement all three functions in the code editor (see the Reference section below):
```python
def wrap_key(dek: bytes, kek: bytes) -> bytes
def unwrap_key(wrapped: bytes, kek: bytes) -> bytes
def looks_like_csprng(key_bytes: bytes) -> bool
def detect_ecb_leakage(ciphertext_blocks: list) -> bool
```

## Input
- `wrap_key`/`unwrap_key`: `dek`/`wrapped` are `bytes` (may be empty); `kek` is non-empty `bytes`.
- `looks_like_csprng`: `key_bytes` is `bytes`, at least 1 byte.
- `detect_ecb_leakage`: `ciphertext_blocks` is `list[bytes]`, all blocks the same length.

## Output
- `wrap_key`/`unwrap_key` return `bytes` the same length as their key argument.
- `looks_like_csprng` returns `bool`.
- `detect_ecb_leakage` returns `bool`.

## Examples
```python
wrap_key(b"\x01\x02\x03\x04", b"\xff\x00")
# -> bytes([0x01^0xff, 0x02^0x00, 0x03^0xff, 0x04^0x00]) == b"\xfe\x02\xfc\x04"

unwrap_key(wrap_key(dek, kek), kek) == dek   # always, for any non-empty kek

looks_like_csprng(bytes([5]*16))            -> False   # single repeated byte
looks_like_csprng(bytes(range(16)))         -> False   # arithmetic/monotonic sequence
looks_like_csprng(b"\x4e\x91\x0a\xd3\x77\xbb\x02\xf8")  -> True   # no obvious structure

detect_ecb_leakage([b"AAAAAAAA", b"BBBBBBBB", b"AAAAAAAA"]) -> True    # block 0 == block 2
detect_ecb_leakage([b"AAAAAAAA", b"BBBBBBBB", b"CCCCCCCC"]) -> False
```

## Constraints
- Deterministic, offline, standard library only. No `import random`, no real crypto libraries
  (`cryptography`, `pycryptodome`, `hashlib`-based ciphers, etc.).
- `kek` is always non-empty for `wrap_key`/`unwrap_key`; `dek` may be empty (returns `b""`).
- `looks_like_csprng` receives at least 1 byte; treat any input shorter than 8 bytes as
  "too short to judge" and return `False` (not enough data to look random).
- `detect_ecb_leakage` receives zero or more equal-length `bytes` blocks.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry points
```python
def wrap_key(dek: bytes, kek: bytes) -> bytes
def unwrap_key(wrapped: bytes, kek: bytes) -> bytes
def looks_like_csprng(key_bytes: bytes) -> bool
def detect_ecb_leakage(ciphertext_blocks: list) -> bool
```

### Input / output

#### `wrap_key(dek, kek) -> bytes`
- `dek`: `bytes`, the plaintext Data Encryption Key to wrap (may be empty).
- `kek`: `bytes`, the Key Encryption Key, non-empty.
- Returns `bytes` of the same length as `dek`, where output byte `i` is
  `dek[i] XOR kek[i % len(kek)]` (repeating-XOR keystream).

#### `unwrap_key(wrapped, kek) -> bytes`
- Inverse of `wrap_key`: `unwrap_key(wrap_key(dek, kek), kek) == dek` for any `dek` and any
  non-empty `kek`, since XOR is its own inverse.

#### `looks_like_csprng(key_bytes) -> bool`
- `key_bytes`: `bytes`, a candidate generated key.
- Returns `False` if `len(key_bytes) < 8` (too short to judge).
- Returns `False` if the bytes show obvious low-entropy structure:
  - all bytes identical, OR
  - the byte sequence is a short repeating pattern (period `p` with
    `1 <= p <= len(key_bytes)//2`) that tiles the whole input, OR
  - the byte sequence is strictly a constant-step arithmetic sequence (e.g. `0,1,2,3...` or
    `10,8,6,4...`), taken modulo 256.
- Returns `True` otherwise.

#### `detect_ecb_leakage(ciphertext_blocks) -> bool`
- `ciphertext_blocks`: `list[bytes]`, all the same length (as ECB-mode block-cipher output would
  be); may be empty or contain a single block.
- Returns `True` if any two blocks in the list are byte-for-byte identical (indicating the
  plaintext blocks that produced them were also identical, the ECB pattern-leakage weakness).
- Returns `False` if all blocks are distinct, or if there are fewer than 2 blocks.

### Guarantees / rules
- Pure functions, deterministic, no I/O, no network. Standard library only.
- No use of `random`, `os.urandom`, `secrets`, or any real cipher/hash primitive, the point of
  this toy toolkit is to reason about structure and wrapping mechanics, not to be secure.
- `wrap_key`/`unwrap_key`: `kek` non-empty is a caller guarantee, not something the function must
  validate defensively beyond raising on empty `kek`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 5.3.b. Student completes the TODOs.
This file intentionally FAILS the tests until implemented."""


def wrap_key(dek: bytes, kek: bytes) -> bytes:
    """Wrap a Data Encryption Key (DEK) with a Key Encryption Key (KEK) using a
    repeating-XOR keystream (toy envelope encryption — NOT real crypto).

    Args:
        dek: plaintext key to wrap (may be empty).
        kek: non-empty wrapping key.
    Returns:
        bytes of len(dek), where byte i = dek[i] XOR kek[i % len(kek)].
    """
    # TODO: XOR each byte of dek with the KEK byte at the same position, wrapping
    #       the KEK index around with modulo len(kek).
    raise NotImplementedError("implement wrap_key() — see the problem statement above")


def unwrap_key(wrapped: bytes, kek: bytes) -> bytes:
    """Inverse of wrap_key(): recovers the original DEK given the same KEK.

    Args:
        wrapped: bytes produced by wrap_key(dek, kek).
        kek: the same non-empty KEK used to wrap.
    Returns:
        the original dek bytes.
    """
    # TODO: XOR is its own inverse — this can reuse the same logic as wrap_key.
    raise NotImplementedError("implement unwrap_key() — see the problem statement above")


def looks_like_csprng(key_bytes: bytes) -> bool:
    """Heuristically flag whether key_bytes could plausibly have come from a
    CSPRNG, vs. showing obvious low-entropy structure (repeated byte, short
    repeating pattern, or an arithmetic/monotonic sequence).

    Args:
        key_bytes: candidate generated key.
    Returns:
        False if len(key_bytes) < 8 (too short to judge), or if the bytes show
        obvious structure. True otherwise.
    """
    # TODO: check length < 8 -> False.
    # TODO: check all-identical bytes -> False.
    # TODO: check for a short repeating pattern (period 1..len//2) that tiles
    #       the whole input -> False.
    # TODO: check for a constant-step arithmetic sequence (mod 256) -> False.
    # TODO: otherwise -> True.
    raise NotImplementedError("implement looks_like_csprng() — see the problem statement above")


def detect_ecb_leakage(ciphertext_blocks: list) -> bool:
    """Detect ECB-mode pattern leakage: True if any two blocks are identical.

    Args:
        ciphertext_blocks: list[bytes], all the same length.
    Returns:
        True if any duplicate block exists, False otherwise (or if < 2 blocks).
    """
    # TODO: compare blocks pairwise (or use a set) to find any duplicate.
    raise NotImplementedError("implement detect_ecb_leakage() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- wrap_key / unwrap_key -----------------------------------------------
def test_wrap_key_basic_xor():
    dek = b"\x01\x02\x03\x04"
    kek = b"\xff\x00"
    assert impl.wrap_key(dek, kek) == bytes([0x01 ^ 0xFF, 0x02 ^ 0x00, 0x03 ^ 0xFF, 0x04 ^ 0x00])


def test_wrap_unwrap_round_trip():
    dek = b"top-secret-dek-1234"
    kek = b"kekmaterial"
    wrapped = impl.wrap_key(dek, kek)
    assert wrapped != dek
    assert impl.unwrap_key(wrapped, kek) == dek


def test_wrap_key_empty_dek():
    assert impl.wrap_key(b"", b"anykek") == b""


def test_wrap_key_kek_shorter_than_dek_repeats():
    dek = b"\x00\x00\x00\x00"
    kek = b"\x01\x02"
    assert impl.wrap_key(dek, kek) == b"\x01\x02\x01\x02"


# ---- looks_like_csprng -----------------------------------------------------
def test_repeated_byte_is_weak():
    assert impl.looks_like_csprng(bytes([5] * 16)) is False


def test_arithmetic_sequence_is_weak():
    assert impl.looks_like_csprng(bytes(range(16))) is False


def test_short_repeating_pattern_is_weak():
    assert impl.looks_like_csprng(b"abab" * 4) is False


def test_too_short_is_weak():
    assert impl.looks_like_csprng(b"\x01\x02\x03") is False


def test_plausible_csprng_output_is_strong():
    assert impl.looks_like_csprng(b"\x4e\x91\x0a\xd3\x77\xbb\x02\xf8") is True


def test_another_plausible_csprng_output():
    assert impl.looks_like_csprng(bytes([0x9F, 0x02, 0x77, 0xC1, 0x3A, 0x5E, 0x00, 0xE4, 0x6B])) is True


# ---- detect_ecb_leakage -----------------------------------------------------
def test_duplicate_blocks_flagged():
    blocks = [b"AAAAAAAA", b"BBBBBBBB", b"AAAAAAAA"]
    assert impl.detect_ecb_leakage(blocks) is True


def test_all_distinct_blocks_not_flagged():
    blocks = [b"AAAAAAAA", b"BBBBBBBB", b"CCCCCCCC"]
    assert impl.detect_ecb_leakage(blocks) is False


def test_single_block_not_flagged():
    assert impl.detect_ecb_leakage([b"AAAAAAAA"]) is False


def test_empty_blocks_not_flagged():
    assert impl.detect_ecb_leakage([]) is False


def test_duplicate_at_end_of_longer_list():
    blocks = [b"11111111", b"22222222", b"33333333", b"44444444", b"22222222"]
    assert impl.detect_ecb_leakage(blocks) is True


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
